# Homework 6.1 - Step 1

## Attention for Tokenized Phase-Space Dynamics

### Choice of Dynamical System: The Lorenz System

For Homework 6.1, I continue with the **Lorenz system** from Homework 5.2.

The Lorenz equations are

$$
\dot{x}=\sigma(y-x),
$$

$$
\dot{y}=x(\rho-z)-y,
$$

$$
\dot{z}=xy-\beta z,
$$

with the classical parameter values

$$
\sigma=10,
\qquad
\rho=28,
\qquad
\beta=\frac{8}{3}.
$$

This is a useful choice because Homework 6.1 asks us to return to the tokenized dynamical systems from the previous chapter and replace the fixed-memory Markov or m-gram predictor with an attention-based next-token predictor.

Using the same Lorenz system lets us keep the underlying dynamical data and tokenization framework approximately fixed while changing the prediction model. This makes the later comparison between the Markov baseline and the attention model much easier to interpret.

## Why the Lorenz System Is a Good Choice

The Lorenz system is especially suitable for this project for several reasons.

First, it is low-dimensional. The state is only

$$
X(t)=(x(t),y(t),z(t)),
$$

so trajectories are straightforward to simulate, discretize, and visualize.

Second, the system is chaotic for the classical parameter values. Nearby initial conditions can eventually produce very different trajectories, even though the equations are completely deterministic.

Third, the Lorenz trajectory repeatedly moves around two main lobes of the attractor. This gives the token sequence nontrivial temporal structure. A model can be tested not only on next-token accuracy, but also on whether generated sequences reproduce the characteristic switching behavior between the two lobes.

Finally, we already used the Lorenz system in Homework 5.2. This gives us a natural baseline:

$$
\text{Lorenz tokens + fixed-memory Markov model}
$$

versus

$$
\text{Lorenz tokens + learned attention mechanism}.
$$

Therefore, the main change in Homework 6.1 is the prediction mechanism rather than the underlying physical system.

## Mathematical Classification of the Lorenz System

Let

$$
F(x,y,z)
=
\left(
\sigma(y-x),
x(\rho-z)-y,
xy-\beta z
\right).
$$

The divergence is

$$
\nabla\cdot F
=
\frac{\partial \dot{x}}{\partial x}
+
\frac{\partial \dot{y}}{\partial y}
+
\frac{\partial \dot{z}}{\partial z}.
$$

For the Lorenz equations,

$$
\frac{\partial \dot{x}}{\partial x}
=
-\sigma,
$$

$$
\frac{\partial \dot{y}}{\partial y}
=
-1,
$$

and

$$
\frac{\partial \dot{z}}{\partial z}
=
-\beta.
$$

Therefore,

$$
\nabla\cdot F
=
-\sigma-1-\beta.
$$

Using the classical parameters,

$$
\nabla\cdot F
=
-10-1-\frac{8}{3}
=
-\frac{41}{3}<0.
$$

Thus the Lorenz system is **dissipative**.

The negative divergence means that phase-space volumes contract as time evolves. This is consistent with trajectories approaching a lower-dimensional attracting set rather than filling all of three-dimensional phase space.

For the parameter values used here, the Lorenz system is also **chaotic**. It is deterministic, but it exhibits sensitive dependence on initial conditions.

It is not treated as a Hamiltonian system in this project, and in this classical chaotic regime it is not treated as an integrable system determined by a sufficient family of conserved quantities.

In [4]:
sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0

divergence = -sigma - 1.0 - beta

print("Lorenz parameters")
print("sigma =", sigma)
print("rho   =", rho)
print("beta  =", beta)
print()
print("Divergence =", divergence)
print("Dissipative:", divergence < 0)

Lorenz parameters
sigma = 10.0
rho   = 28.0
beta  = 2.6666666666666665

Divergence = -13.666666666666666
Dissipative: True


# What Attention Means for a Tokenized Dynamical System

In Homework 5.2, each continuous Lorenz trajectory

$$
X(t_1),X(t_2),\ldots,X(t_N)
$$

was converted into a token sequence

$$
a_1,a_2,\ldots,a_N,
$$

where each token records which phase-space box contains the state at that sampled time.

A Markov or m-gram predictor uses a fixed number of previous tokens.

For example, an order-3 model predicts

$$
P(a_{n+1}\mid a_{n-2},a_{n-1},a_n).
$$

The model always uses exactly the most recent three tokens.

An attention model is different. It can be given a longer history

$$
a_{n-L+1},\ldots,a_n
$$

and learn which previous positions are most useful for predicting

$$
a_{n+1}.
$$

The basic idea is

$$
\boxed{\text{Markov model: fixed memory}}
$$

versus

$$
\boxed{\text{Attention model: learned data-dependent memory}}.
$$

## Why Attention Can Help After Phase-Space Tokenization

The original Lorenz equations are first-order differential equations.

If the exact continuous state

$$
X(t)
$$

were known perfectly, then the immediate future is determined locally by the vector field

$$
\dot{X}(t)=F(X(t)).
$$

However, after phase space is divided into boxes, a token tells us only that the state lies somewhere inside one box.

Suppose

$$
a_n=731.
$$

This does not reveal the exact location of the continuous state inside box 731, nor does it reveal the direction from which the trajectory entered that box.

Two different histories can therefore end at the same current token:

$$
\ldots,415,522,731
$$

and

$$
\ldots,902,811,731,
$$

while corresponding to different continuous states and different directions of motion.

Their next tokens may be different.

Thus the tokenization introduces partial loss of state information. Earlier tokens can help recover some information about how the trajectory arrived at the current box.

This is the main reason attention may be useful in the tokenized Lorenz problem.

# Basic Attention Model for the Lorenz Tokens

The attention model used later in Homework 6.1 should contain the components requested in the assignment:

1. token embeddings;
2. positional information;
3. one causal self-attention layer;
4. a softmax output for next-token prediction.

## Token embeddings

Each discrete phase-space token is mapped to a learned vector:

$$
a_j
\longrightarrow
e_j\in\mathbb{R}^d.
$$

## Positional information

Because a trajectory is ordered in time, the model must know which token occurred earlier or later.

We therefore combine token information with positional information:

$$
h_j=e_j+p_j.
$$

## Queries, keys, and values

From each representation,

$$
q_i=W_Qh_i,
\qquad
k_j=W_Kh_j,
\qquad
v_j=W_Vh_j.
$$

## Causal attention weights

For a current position $$i$$, attention is allowed to use only positions

$$
j\le i.
$$

A standard score is

$$
s_{ij}
=
\frac{q_i^\top k_j}{\sqrt{d_k}}.
$$

After applying the causal mask and softmax,

$$
A_{ij}
=
\frac{\exp(s_{ij})}
{\sum_{r\le i}\exp(s_{ir})}.
$$

The weights satisfy

$$
A_{ij}\ge0,
$$

and

$$
\sum_{j\le i}A_{ij}=1.
$$

The attention output is

$$
o_i
=
\sum_{j\le i}
A_{ij}v_j.
$$

The final output layer converts this representation into a probability distribution over the next phase-space token:

$$
P_\theta
\left(
a_{i+1}
\mid
a_1,\ldots,a_i
\right).
$$

# Main Question for Homework 6.1

The previous homework asked whether a fixed amount of recent token history could predict the next phase-space token.

Homework 6.1 asks a more flexible question:

$$
\boxed{
\text{Can attention learn which parts of the past trajectory matter for next-token prediction?}
}
$$

For the Lorenz system, we will later investigate:

- whether attention improves next-token prediction relative to the Markov or m-gram baseline;
- whether the improvement depends on context length;
- whether attention concentrates mostly on recent states or sometimes uses earlier states;
- whether attention changes near transitions between the two Lorenz lobes;
- whether generated token trajectories remain in the coarse Lorenz attractor region;
- whether generated sequences reproduce the two-lobe switching behavior.

An important interpretation rule is that improved next-token accuracy does not automatically imply that the attention model has recovered the true Lorenz dynamics.

The attention weights must be analyzed separately to determine whether they correspond to meaningful dynamical information or simply improve short-term prediction.

## Step 1 Conclusion

The Lorenz system will be used for Homework 6.1 because it provides a direct continuation of Homework 5.2 and a controlled comparison between fixed-memory and learned-memory sequence models.

The system is

- deterministic,
- dissipative,
- chaotic,
- low-dimensional,
- easy to visualize,
- and well suited for studying whether attention learns meaningful temporal dependencies.

The next step is to generate or reuse the Lorenz trajectory collection, apply the same phase-space discretization framework, and construct the token sequences that will be used by both the baseline predictor and the attention model.